# 🏏 IPL Cricket Data Analytics Dashboard
### End-to-End Data Analytics Project
**Dataset:** IPL Ball-by-Ball Dataset (IPL.csv)  
**Tools:** Python · Pandas · NumPy · Plotly · Streamlit  
**Project:** IBM Bob Project

---
## 📦 Step 0 — Install & Import Libraries

In [ ]:
# Install required libraries (run once)
# !pip install pandas numpy plotly streamlit

import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print('✅ All libraries imported successfully')
print(f'   Pandas  : {pd.__version__}')
print(f'   NumPy   : {np.__version__}')

---
## 📂 Step 1 — Load & Explore the Dataset

In [ ]:
# Load the dataset
# Adjust the path if running from a different directory
DATA_PATH = 'data/IPL.csv'

df = pd.read_csv(DATA_PATH, low_memory=False)

print('=== DATASET OVERVIEW ===')
print(f'Total Rows    : {len(df):,}')
print(f'Total Columns : {len(df.columns)}')

In [ ]:
# View first 5 rows
df.head()

In [ ]:
# All column names
print('=== COLUMN NAMES (64 total) ===')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
# Data types — numerical vs categorical
print('=== NUMERICAL COLUMNS ===')
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(num_cols)

print('\n=== CATEGORICAL COLUMNS ===')
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(cat_cols)

In [ ]:
# Missing value analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
print('=== MISSING VALUES ===')
print(missing_df.to_string())

In [ ]:
# Duplicate check
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Unique match IDs: {df["match_id"].nunique():,}')
print(f'Unique batters: {df["batter"].nunique():,}')
print(f'Unique bowlers: {df["bowler"].nunique():,}')
print(f'Seasons: {sorted(df["year"].dropna().unique().astype(int).tolist())}')

In [ ]:
# Data Dictionary
data_dict = {
    'match_id':        'Unique identifier for each match',
    'date':            'Date of the match',
    'season':          'IPL season label (e.g. 2007/08)',
    'batting_team':    'Team currently batting',
    'bowling_team':    'Team currently bowling',
    'innings':         'Innings number (1 or 2)',
    'over':            'Over number within innings',
    'batter':          'Batter facing the delivery',
    'runs_batter':     'Runs scored by batter off this delivery',
    'bowler':          'Bowler delivering the ball',
    'valid_ball':      '1 = legal delivery counting toward over',
    'runs_extras':     'Extra runs (wides, no-balls, byes, leg-byes)',
    'runs_total':      'Total runs from delivery (batter + extras)',
    'runs_bowler':     'Runs charged to the bowler',
    'wicket_kind':     'Type of dismissal (if any)',
    'player_out':      'Batter dismissed (if any)',
    'toss_winner':     'Team that won the toss',
    'toss_decision':   'bat or field',
    'match_won_by':    'Team that won the match',
    'win_outcome':     'Margin of victory',
    'venue':           'Stadium name',
    'player_of_match': 'Player of the Match award',
}
dd_df = pd.DataFrame(list(data_dict.items()), columns=['Column', 'Description'])
dd_df

---
## 🧹 Step 2 — Data Cleaning

In [ ]:
# 2.1 Parse dates and derive season year
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['season_year'] = df['date'].dt.year.fillna(df['year']).astype('Int64')
print('✅ Dates parsed')

# 2.2 Remove exact duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f'✅ Removed {before - len(df)} duplicate rows')

# 2.3 Numeric coercions
num_cols = ['over','ball','ball_no','bat_pos','runs_batter','balls_faced',
            'valid_ball','runs_extras','runs_total','runs_bowler',
            'runs_not_boundary','non_striker_pos','team_runs','team_balls',
            'team_wicket','batter_runs','batter_balls','bowler_wicket','innings']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
print('✅ Numeric columns coerced')

# 2.4 Boundary flags
df['is_four'] = ((df['runs_batter'] == 4) & (df['runs_not_boundary'] == 0)).astype(int)
df['is_six']  = ((df['runs_batter'] == 6) & (df['runs_not_boundary'] == 0)).astype(int)
print('✅ Boundary flags created')

# 2.5 Wicket flag
df['is_wicket'] = df['wicket_kind'].notna() & (df['wicket_kind'].str.strip() != '')
print('✅ Wicket flag created')

# 2.6 Filter to innings 1 & 2 only
df_main = df[df['innings'].isin([1, 2])].copy()
print(f'✅ Filtered to main innings: {len(df_main):,} rows')

In [ ]:
# Verify cleaned data
print('=== POST-CLEANING SUMMARY ===')
print(f'Rows in df_main   : {len(df_main):,}')
print(f'Total Fours       : {df_main["is_four"].sum():,}')
print(f'Total Sixes       : {df_main["is_six"].sum():,}')
print(f'Total Wickets     : {df_main["is_wicket"].sum():,}')
print(f'Null in batter    : {df_main["batter"].isnull().sum()}')
print(f'Null in bowler    : {df_main["bowler"].isnull().sum()}')

---
## 📊 Step 3 — Data Analysis
### 3.1 Overall KPIs

In [ ]:
# Match-level frame
match_cols = ['match_id','date','season_year','season','batting_team','bowling_team',
              'venue','city','toss_winner','toss_decision','match_won_by',
              'win_outcome','result_type','player_of_match']
match_cols = [c for c in match_cols if c in df.columns]
matches_df = df.drop_duplicates('match_id')[match_cols].copy()

total_matches  = matches_df['match_id'].nunique()
total_seasons  = matches_df['season_year'].nunique()
total_teams    = pd.unique(pd.concat([df_main['batting_team'], df_main['bowling_team']]).dropna()).size
total_players  = pd.unique(pd.concat([df_main['batter'], df_main['bowler']]).dropna()).size
total_runs     = int(df_main['runs_total'].sum())
total_wickets  = int(df_main['is_wicket'].sum())
total_balls    = int(df_main['valid_ball'].sum())

print('=' * 40)
print('       IPL OVERALL KPIs')
print('=' * 40)
print(f'Total Matches       : {total_matches:,}')
print(f'Total Seasons       : {total_seasons}')
print(f'Total Teams         : {total_teams}')
print(f'Total Players       : {total_players}')
print(f'Total Runs          : {total_runs:,}')
print(f'Total Wickets       : {total_wickets:,}')
print(f'Total Valid Balls   : {total_balls:,}')

### 3.2 Season Analysis

In [ ]:
season_analysis = (
    df_main.groupby('season_year')
    .agg(
        total_runs    = ('runs_total', 'sum'),
        total_wickets = ('is_wicket',  'sum'),
        total_balls   = ('valid_ball', 'sum'),
        matches       = ('match_id',   'nunique')
    )
    .reset_index()
)
season_analysis['avg_runs_per_match'] = (season_analysis['total_runs'] / season_analysis['matches']).round(1)
season_analysis.sort_values('season_year')

### 3.3 Team Analysis

In [ ]:
win_map = matches_df[matches_df['result_type'] != 'no result'].copy() if 'result_type' in matches_df.columns else matches_df.copy()

team_wins = (
    win_map.groupby('match_won_by')['match_id'].nunique()
    .reset_index()
    .rename(columns={'match_id': 'wins', 'match_won_by': 'team'})
    .sort_values('wins', ascending=False)
)

bat_matches = df_main.groupby('batting_team')['match_id'].nunique().reset_index().rename(
    columns={'batting_team': 'team', 'match_id': 'matches_played'})
team_wins = team_wins.merge(bat_matches, on='team', how='left')
team_wins['losses']  = team_wins['matches_played'] - team_wins['wins']
team_wins['win_pct'] = (team_wins['wins'] / team_wins['matches_played'] * 100).round(1)

print('=== TEAM WINS TABLE ===')
team_wins

### 3.4 Batting Analysis

In [ ]:
batting = (
    df_main.groupby('batter')
    .agg(
        total_runs     = ('runs_batter', 'sum'),
        balls_faced    = ('valid_ball',  'sum'),
        fours          = ('is_four',     'sum'),
        sixes          = ('is_six',      'sum'),
        innings_played = ('match_id',    'nunique'),
    )
    .reset_index()
)
batting['strike_rate'] = (batting['total_runs'] / batting['balls_faced'] * 100).round(2)
batting['strike_rate'] = batting['strike_rate'].replace([np.inf, -np.inf], 0)
batting.sort_values('total_runs', ascending=False, inplace=True)

print('=== TOP 15 RUN SCORERS ===')
batting[['batter','total_runs','balls_faced','strike_rate','fours','sixes','innings_played']].head(15)

In [ ]:
# Highest individual innings
highest_scores = (
    df_main.groupby(['match_id','batter'])
    .agg(score=('runs_batter','sum'), balls=('valid_ball','sum'))
    .reset_index()
    .sort_values('score', ascending=False)
)
print('=== HIGHEST INDIVIDUAL INNINGS ===')
highest_scores.head(10)

### 3.5 Bowling Analysis

In [ ]:
bowling = (
    df_main.groupby('bowler')
    .agg(
        wickets       = ('is_wicket',   'sum'),
        balls_bowled  = ('valid_ball',  'sum'),
        runs_conceded = ('runs_bowler', 'sum'),
        matches       = ('match_id',    'nunique'),
    )
    .reset_index()
)
bowling['economy'] = (bowling['runs_conceded'] / (bowling['balls_bowled'] / 6)).round(2)
bowling['economy'] = bowling['economy'].replace([np.inf, -np.inf], 0)
bowling['avg']     = np.where(bowling['wickets'] > 0,
                               (bowling['runs_conceded'] / bowling['wickets']).round(2), np.nan)
bowling.sort_values('wickets', ascending=False, inplace=True)

print('=== TOP 15 WICKET TAKERS ===')
bowling[['bowler','wickets','economy','avg','balls_bowled','matches']].head(15)

### 3.6 Venue Analysis

In [ ]:
venue_analysis = (
    df_main.groupby('venue')
    .agg(
        matches       = ('match_id',   'nunique'),
        total_runs    = ('runs_total', 'sum'),
        total_wickets = ('is_wicket',  'sum'),
    )
    .reset_index()
    .sort_values('matches', ascending=False)
)
venue_analysis['avg_runs_per_match'] = (venue_analysis['total_runs'] / venue_analysis['matches']).round(1)
print('=== TOP 15 VENUES ===')
venue_analysis.head(15)

### 3.7 Toss Analysis

In [ ]:
toss_df = matches_df.dropna(subset=['toss_winner','match_won_by']).copy()
toss_df['toss_win_match_win'] = (toss_df['toss_winner'] == toss_df['match_won_by']).astype(int)

toss_decision_counts = toss_df['toss_decision'].value_counts()
toss_win_pct = toss_df['toss_win_match_win'].mean() * 100

print('=== TOSS ANALYSIS ===')
print(f'Toss Decision Counts:\n{toss_decision_counts.to_string()}')
print(f'\nToss winner won the match: {toss_df["toss_win_match_win"].sum()} times')
print(f'Win % after winning toss : {toss_win_pct:.1f}%')

toss_outcome = (
    toss_df.groupby('toss_decision')['toss_win_match_win']
    .agg(wins='sum', total='count')
    .reset_index()
)
toss_outcome['win_pct'] = (toss_outcome['wins'] / toss_outcome['total'] * 100).round(1)
toss_outcome

---
## 💡 Step 4 — Key Insights

In [ ]:
top_bat  = batting.iloc[0]
top_bowl = bowling.iloc[0]
top_team = team_wins.iloc[0]
top_hs   = highest_scores.iloc[0]

potm = (
    matches_df.dropna(subset=['player_of_match'])
    .groupby('player_of_match')['match_id'].count()
    .reset_index()
    .rename(columns={'match_id':'awards','player_of_match':'player'})
    .sort_values('awards', ascending=False)
)

print('=' * 55)
print('            KEY INSIGHTS FROM IPL DATA')
print('=' * 55)
print(f"  1. Top Run Scorer     : {top_bat['batter']} — {top_bat['total_runs']:,} runs")
print(f"  2. Top Wicket Taker   : {top_bowl['bowler']} — {top_bowl['wickets']} wickets")
print(f"  3. Most Wins (Team)   : {top_team['team']} — {top_team['wins']} wins ({top_team['win_pct']}%)")
print(f"  4. Highest Innings    : {top_hs['score']} runs by {top_hs['batter']}")
print(f"  5. Most POTM Awards   : {potm.iloc[0]['player']} — {potm.iloc[0]['awards']} awards")
print(f"  6. Toss Advantage     : Toss winners win {toss_win_pct:.1f}% of matches")
print(f"  7. Total Runs Scored  : {total_runs:,} across {total_matches} matches")
print(f"  8. Most Active Venue  : {venue_analysis.iloc[0]['venue']} ({venue_analysis.iloc[0]['matches']} matches)")
print(f"  9. Avg Runs/Match     : {total_runs // total_matches} runs")
print(f" 10. Highest SR Batter  : {batting[batting['balls_faced']>=200].sort_values('strike_rate',ascending=False).iloc[0]['batter']}")

---
## 📈 Step 5 — Data Visualizations
### 5.1 Season-wise Run Trend

In [ ]:
fig = px.line(
    season_analysis.sort_values('season_year'),
    x='season_year', y='total_runs',
    markers=True,
    title='📈 Total Runs per IPL Season',
    labels={'season_year': 'Season Year', 'total_runs': 'Total Runs'},
    color_discrete_sequence=['#FF6B35']
)
fig.update_layout(template='plotly_dark')
fig.show()

### 5.2 Average Runs per Match by Season

In [ ]:
fig = px.bar(
    season_analysis.sort_values('season_year'),
    x='season_year', y='avg_runs_per_match',
    title='🏏 Average Runs per Match by Season',
    labels={'season_year': 'Season Year', 'avg_runs_per_match': 'Avg Runs/Match'},
    color='avg_runs_per_match', color_continuous_scale='Oranges'
)
fig.update_layout(template='plotly_dark')
fig.show()

### 5.3 Team Wins Comparison

In [ ]:
fig = px.bar(
    team_wins.sort_values('wins', ascending=True),
    x='wins', y='team',
    orientation='h',
    title='🏆 Total Wins per Team',
    labels={'wins': 'Wins', 'team': 'Team'},
    color='wins', color_continuous_scale='Reds'
)
fig.update_layout(template='plotly_dark', height=500)
fig.show()

### 5.4 Top 10 Run Scorers

In [ ]:
top10_bat = batting.head(10)
fig = px.bar(
    top10_bat.sort_values('total_runs', ascending=True),
    x='total_runs', y='batter',
    orientation='h',
    title='🏏 Top 10 Run Scorers (All IPL Seasons)',
    labels={'total_runs': 'Total Runs', 'batter': 'Batter'},
    color='total_runs', color_continuous_scale='YlOrRd'
)
fig.update_layout(template='plotly_dark')
fig.show()

### 5.5 Top 10 Wicket Takers

In [ ]:
top10_bowl = bowling.head(10)
fig = px.bar(
    top10_bowl.sort_values('wickets', ascending=True),
    x='wickets', y='bowler',
    orientation='h',
    title='🎯 Top 10 Wicket Takers (All IPL Seasons)',
    labels={'wickets': 'Wickets', 'bowler': 'Bowler'},
    color='wickets', color_continuous_scale='Purples'
)
fig.update_layout(template='plotly_dark')
fig.show()

### 5.6 Fours vs Sixes — Top Boundary Hitters

In [ ]:
top10_bound = batting.head(10)[['batter','fours','sixes']].melt(
    id_vars='batter', var_name='type', value_name='count')
fig = px.bar(
    top10_bound, x='batter', y='count', color='type',
    barmode='group',
    title='🏏 Fours vs Sixes — Top 10 Batsmen',
    color_discrete_map={'fours': '#2196F3', 'sixes': '#9C27B0'}
)
fig.update_layout(template='plotly_dark')
fig.show()

### 5.7 Toss Decision Distribution

In [ ]:
toss_dec = toss_df['toss_decision'].value_counts().reset_index()
toss_dec.columns = ['decision', 'count']
fig = px.pie(
    toss_dec, names='decision', values='count',
    title='🎲 Toss Decision Distribution (Bat vs Field)',
    color_discrete_sequence=['#FF6B35','#4CAF50']
)
fig.update_layout(template='plotly_dark')
fig.show()

### 5.8 Top Venues by Matches Hosted

In [ ]:
fig = px.bar(
    venue_analysis.head(15).sort_values('matches', ascending=True),
    x='matches', y='venue',
    orientation='h',
    title='🏟️ Top 15 Venues by Matches Hosted',
    labels={'matches': 'Matches', 'venue': 'Venue'},
    color='matches', color_continuous_scale='Blues'
)
fig.update_layout(template='plotly_dark', height=500)
fig.show()

### 5.9 Runs vs Strike Rate Scatter (Batsmen)

In [ ]:
scatter_df = batting[batting['balls_faced'] >= 100].copy()
fig = px.scatter(
    scatter_df, x='total_runs', y='strike_rate',
    hover_name='batter', size='balls_faced', color='sixes',
    color_continuous_scale='Oranges',
    title='🏏 Runs vs Strike Rate (bubble = balls faced, colour = sixes)'
)
fig.update_layout(template='plotly_dark')
fig.show()

### 5.10 Wins vs Losses per Team

In [ ]:
fig = go.Figure()
tw = team_wins.sort_values('wins', ascending=False).head(12)
fig.add_trace(go.Bar(name='Wins',   x=tw['team'], y=tw['wins'],   marker_color='#4CAF50'))
fig.add_trace(go.Bar(name='Losses', x=tw['team'], y=tw['losses'], marker_color='#F44336'))
fig.update_layout(
    barmode='group',
    title='🏆 Wins vs Losses — Top 12 Teams',
    template='plotly_dark'
)
fig.show()

---
## 🚀 Step 6 — Launch the Streamlit Dashboard

Run the following command from a **terminal** inside the `ipl_cricket_analytics/` folder:

```bash
# Step 1: Generate processed data (already done if you ran this notebook)
python analysis/data_analysis.py

# Step 2: Launch the dashboard
streamlit run frontend/app.py
```

Then open your browser at **http://localhost:8501**

---
## ✅ Project Complete

| Component | Status |
|-----------|--------|
| Data Exploration | ✅ |
| Data Cleaning | ✅ |
| Overall KPIs | ✅ |
| Team Analysis | ✅ |
| Batting Analysis | ✅ |
| Bowling Analysis | ✅ |
| Venue Analysis | ✅ |
| Toss Analysis | ✅ |
| 10 Visualizations | ✅ |
| Streamlit Dashboard | ✅ |